# 05 - Model evaluation

Purpose: score the trained model properly and report what it can and cannot do.

Two score families are used because they answer different questions:

* **continuous** (MAE, RMSE, bias, NSE) - millimetre error, relevant to a drainage or
  crop advisor;
* **categorical** (POD, FAR, CSI, Brier) - heavy-rain event detection at 15 mm/day,
  relevant to an alert consumer.

Rainfall is intermittent, so continuous scores are dominated by dry days. The
categorical breakdown is reported alongside them, never instead of them.

In [ ]:
import sys
from pathlib import Path

# The notebook kernel runs with the notebook directory as the working directory,
# but paths are resolved from the repository root so cells behave the same when
# executed by nbconvert, JupyterLab or an IDE.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "ml" / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

ML_SRC = REPO_ROOT / "ml" / "src"
if str(ML_SRC) not in sys.path:
    sys.path.insert(0, str(ML_SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = REPO_ROOT / "data" / "processed"
ARTIFACT_ROOT = REPO_ROOT / "ml" / "artifacts"
CONFIG_PATH = REPO_ROOT / "ml" / "configs" / "training.yaml"
OBSERVATIONS = DATA_DIR / "observations.csv"
DISTRICTS = DATA_DIR / "districts.csv"

print("notebook environment ready")
print("repo root:", REPO_ROOT)
print("python", sys.version.split()[0])


In [ ]:
import json

from rainml.data.loaders import load_districts, load_observations, attach_coordinates
from rainml.data.cleaning import clean_observations
from rainml.data.validation import chronological_split
from rainml.features.engineering import TARGET_COLUMN, build_training_frame
from rainml.inference.predictor import RainfallPredictor
from rainml.models.registry import list_versions, load_metadata, load_metrics
from rainml.models.forecasting import CANDIDATES, build_pipeline

versions = list_versions(ARTIFACT_ROOT)
print("available versions:", versions)
version = versions[-1]
metadata = load_metadata(ARTIFACT_ROOT, version)
metadata["version"], metadata["features"][:5]

In [ ]:
cleaned, _ = clean_observations(
    attach_coordinates(load_observations(OBSERVATIONS), load_districts(DISTRICTS))
)
frame = build_training_frame(cleaned, horizon_days=1)
train, valid, test = chronological_split(frame)
features = metadata["features"]
print("test rows:", len(test))

## Full error distribution on the test slice

A model that is only accurate on average can still be badly wrong on the days that
matter. The error quantiles below are the honest description of that risk.

In [ ]:
selected_name = metadata["selected_model"]
spec = next(s for s in CANDIDATES if s.name == selected_name)
print("selected model:", selected_name)
print("hyperparameters:", metadata["hyperparameters"])
spec

In [ ]:
pipeline = build_pipeline(spec, features, max_features=14)
pipeline.fit(train[features], train[TARGET_COLUMN].to_numpy(dtype=float))
test_pred = np.clip(pipeline.predict(test[features]), 0, None)
test_obs = test[TARGET_COLUMN].to_numpy(dtype=float)
error = test_obs - test_pred
pd.DataFrame(
    {
        "mean_error_mm": [error.mean()],
        "mean_absolute_error_mm": [np.abs(error).mean()],
        "p50_abs_error": [np.percentile(np.abs(error), 50)],
        "p90_abs_error": [np.percentile(np.abs(error), 90)],
        "p99_abs_error": [np.percentile(np.abs(error), 99)],
        "max_abs_error": [np.abs(error).max()],
    }
).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hexbin(test_obs, test_pred, gridsize=45, cmap="viridis", mincnt=1)
limit = max(test_obs.max(), test_pred.max())
axes[0].plot([0, limit], [0, limit], color="#b91c1c", linestyle="--", label="perfect forecast")
axes[0].set_xlabel("observed (mm)")
axes[0].set_ylabel("predicted (mm)")
axes[0].legend()
axes[0].set_title("Predicted versus observed, test slice")

axes[1].hist(error, bins=70, color="#2563eb")
axes[1].axvline(0, color="#b91c1c", linestyle="--")
axes[1].set_xlabel("error = observed - predicted (mm)")
axes[1].set_title("Error distribution")
plt.tight_layout()
plt.show()

## Event detection at the 15 mm threshold

This is the score an alert consumer should read.

In [ ]:
from rainml.evaluation.metrics import evaluate, evaluate_by_group

scores = evaluate(test_pred, test_obs, threshold_mm=15.0)
pd.Series(scores)

In [ ]:
scored = test.assign(prediction=test_pred, observed=test_obs, absolute_error=np.abs(error))
by_district = evaluate_by_group(scored, observed_column="observed", group_column="district_id")
pd.DataFrame(by_district).T[["sample_size", "mae_mm", "csi", "pod", "far"]].sort_values("mae_mm").round(4)

## Skill against the strongest baseline

Persistence is hard to beat for 24h rainfall. If the model cannot beat it, the honest
conclusion is that the simpler baseline should be served.

In [ ]:
from rainml.models.baseline import make_baseline
from rainml.evaluation.evaluator import score_predictions, skill_score

persistence = make_baseline("persistence")
persistence_pred = persistence.predict(test[features])
model_mae = scores["mae_mm"]
baseline_mae = score_predictions(persistence_pred, test_obs)["mae_mm"]
pd.DataFrame(
    {
        "model_mae_mm": [model_mae],
        "persistence_mae_mm": [baseline_mae],
        "skill_score": [skill_score(model_mae, baseline_mae)],
    }
).round(4)

## Uncertainty interval coverage

An interval that is always too wide is not useful. Empirical coverage of the 10th to
90th percentile band should be near 80%.

In [ ]:
predictor = RainfallPredictor(ARTIFACT_ROOT / version)
features_frame = test[features]
bands = predictor.predict_frame(features_frame)
covered = ((test_obs >= bands["lower_bound_mm"]) & (test_obs <= bands["upper_bound_mm"])).mean()
width = (bands["upper_bound_mm"] - bands["lower_bound_mm"])
pd.DataFrame(
    {
        "nominal_coverage": [0.8],
        "empirical_coverage": [covered],
        "mean_interval_width_mm": [width.mean()],
        "median_interval_width_mm": [width.median()],
    }
).round(4)

## Stability across months

A model that only works in monsoon season is not a general rainfall forecaster.

In [ ]:
scored["month"] = pd.to_datetime(scored["target_date"]).dt.month
monthly = evaluate_by_group(scored, observed_column="observed", group_column="month")
pd.DataFrame(monthly).T[["sample_size", "mae_mm", "csi"]].round(4)

## What this evaluation does not establish

* The dataset is synthetic, so these scores describe the pipeline, not real-world skill.
* The test slice is a single time window; a rolling-origin backtest would give a more
  robust estimate.
* Only a 24h horizon is evaluated. Multi-horizon behaviour is untested.